# Preamble

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import warnings
import keysight.qcs as qcs
import numpy as np
import matplotlib.pyplot as plt
# import sys
# sys.path.append("..")

from keysight.qcs.experiments import ResonatorSpectroscopy, ResonatorSpectroscopy2D, DispersiveShift
from keysight.qcs.experiments import QubitSpectroscopy,RabiExperiment,T1Experiment,RamseyExperiment,IQDistributionExperiment
from keysight.qcs.analysis import FanoResonance, ComplexLorentzian

from preamble.pump_utils import *
from preamble.mapper_utils import *
from preamble.calibration_utils import *
from preamble.pca_gmm_utils import *
from preamble.fitter import *


us,ns,MHz,GHz=1e-6,1e-9,1e6,1e9
pi=np.pi

In [ ]:
cal_file='./config/260702_cal.json'
# set the qubit topology
qubits = qcs.Qudits([0,1,2])
n_qubits=len(qubits)

pairs = [(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)]
n_mq = len(pairs)

topology = qcs.QuditGraph(qubits, pairs)

# set the local oscillator frequency
LO_freq_readout =  6.1*GHz
LO_freq_qubit = 4.3*GHz

# load the calibration variables
variables_from_json = load_json(cal_file)
variables=make_qcs_var(
            variables_from_json=variables_from_json,
            qubits=qubits,
        )

mapper = qcs.ChannelMapper('localhost')
address = load_address("./config/channel_address.yaml")

channel_mapping(mapper,qubits,address)
set_lo_frequencies_RO(mapper,qubits,address,LO_freq_readout)
set_lo_frequencies_XY(mapper,qubits,address,LO_freq_qubit) 
# set_lo_frequencies_XY(mapper,qubits[0],address,4.1*GHz)

backend = qcs.HclBackend(mapper, fpga_postprocessing=True ,init_time=150*us)

cals = MyCalibrationSet(
        topology=topology,
        channels=mapper.channels,
        variables=variables)
cals.export_values(cal_file)

# shorten the variables name
c=cals.variables


print("Backend is ready:", backend.is_system_ready())


In [ ]:
# set the port of windfreak
# port='socket://163.152.38.107:4000' 
port='COM3'
pump = PumpController(port=port).connect()
pump.set(ch=1,freq=7.52*GHz, power=8.4)
pump.close_all()


# Abort program 

In [ ]:
current_id=backend.get_program_execution_history()[0]['accession_id']
backend.abort_program(current_id)

# One-tone spectroscopy (coarse)

In [ ]:
target_qubit=qubits[0]
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.ro_amp,[0.1], index=[0])
set_value_at(c.ro_dur,[5*us], index=[0])
set_value_at(c.acq_dur,[5*us], index=[0])

# Set the frequency range for sweeping
f_start = 6.5*GHz
f_end = 6.8*GHz
n_steps = 301
frequencies = np.linspace(f_start, f_end, n_steps)
# set_lo_frequencies_RO(mapper,qubits,address,f_start-20*MHz) 


one_tone_coarse = ResonatorSpectroscopy(
    backend=backend,
    calibration_set=cals,
    qubits=target_qubit,
    operation="measurement",
)

# Configure the repetitions for this experiment
one_tone_coarse.configure_repetitions(
    frequencies=frequencies, n_shots=500, frequency_name="ro_freq"
)

pump.connect().set_on(ch=1)
one_tone_coarse.execute()
pump.off(ch=1).close_all()

cals.import_values('./config/temp.json')

# Plot the IQ data of the experiment
one_tone_coarse.plot_iq()

In [ ]:
c.ro_freq[0].value = 6.542*GHz
c.ro_freq[1].value = 6.593*GHz
c.ro_freq[2].value = 6.65*GHz

cals.export_values(cal_file)

# One-tone spectroscopy (fine)

In [ ]:
target_qubit=qubits
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.ro_amp,[0.02,0.03,0.03], index=range(n_qubits))
set_value_at(c.ro_dur,[5*us]*n_qubits, index=range(n_qubits))
set_value_at(c.acq_dur,[5*us]*n_qubits, index=range(n_qubits))

# Set the frequency range for sweeping
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 10 * MHz
n_steps = 101
frequencies_2d = np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)


one_tone_fine = ResonatorSpectroscopy(
    backend=backend,
    calibration_set=cals,
    qubits=target_qubit,
    operation="measurement",
)

# Configure the repetitions for this experiment
one_tone_fine.configure_repetitions(
    frequencies=frequencies_2d, n_shots=500, frequency_name="ro_freq"
)

pump.connect().set_on(ch=1)
one_tone_fine.execute()
pump.off(ch=1).close_all()

cals.import_values('./config/temp.json')

In [ ]:
one_tone_fine.fitter=FanoResonance()
ec = one_tone_fine.fit()

print('kappa/2pi:', [round(ec.estimates[k].gamma.val/MHz,2) for k in range(n_qubits)], 'MHz')
print('freq:', [round(ec.estimates[k].resonance.val/GHz,4) for k in range(n_qubits)], 'GHz')
one_tone_fine.plot()

In [ ]:
# f_R0=fit_lorentzian(frequencies_2d[:,0],np.abs(one_tone_fine.get_iq_array(avg=True)[0].to_numpy()))["f0"]
# f_R1=fit_lorentzian(frequencies_2d[:,1],np.abs(one_tone_fine.get_iq_array(avg=True)[1].to_numpy()))["f0"]
# f_R2=fit_lorentzian(frequencies_2d[:,2],np.abs(one_tone_fine.get_iq_array(avg=True)[2].to_numpy()))["f0"]

In [ ]:
one_tone_fine.get_updated_calibration_values()

In [ ]:
c.ro_freq[0].value = 6543372682.684236
c.ro_freq[1].value = 6594239081.3210945
c.ro_freq[2].value = 6652399967.3813095

cals.export_values(cal_file)

# Punch-out experiment

In [ ]:
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.ro_dur,[5*us]*n_qubits, index=range(n_qubits))
set_value_at(c.acq_dur,[5*us]*n_qubits, index=range(n_qubits))


# # Create a resonator spectroscopy experiment
target_qubit=qubits

# Set the frequency range for sweeping
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 10 * MHz
n_steps = 51
frequencies_2d = np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)

# Set the amplitude range for sweeping
dbm_list = np.linspace(-40,-20,5)
ampl_scan_values = 10**(dbm_list/20)
ampl_scan_values_2d = np.tile(ampl_scan_values[:, None], (1, len(target_qubit)))
IQ_list = []


punchout = ResonatorSpectroscopy2D(
    backend=backend,
    calibration_set=cals,
    qubits=target_qubit,
    operation="measurement",
)


# # Configure the repetitions for this experiment
punchout.configure_repetitions(
    frequencies=frequencies_2d,
    frequency_name="ro_freq",
    amplitudes=ampl_scan_values_2d,
    amplitude_name="ro_amp",
    n_shots=1000,
)

pump.connect().set_on(ch=1)
punchout.execute()
pump.off(ch=1).close_all()


for q in range(len(target_qubit)):
    plt.figure(figsize=(7,5))
    for idx, dbm in enumerate(dbm_list):
        plt.plot(
            frequencies_2d[:,q]/GHz,
            20*np.log10(np.abs(punchout.get_iq_array(avg=True)[q,:,idx])),
            label=f"{dbm:.1f} dBm: {round(10**(dbm/20),4)}"
        )
    plt.title('Q'+str(int(target_qubit.labels[q])))
    plt.xlabel("Frequency (GHz)")
    plt.ylabel("Measured amp (dB)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

cals.import_values('./config/temp.json')

# Plot the IQ data as a heatmap
# res_spectroscopy2D.plot_iq(plot_type="2d")

In [ ]:
c.ro_amp[0].value = 0.03
c.ro_amp[1].value = 0.03
c.ro_amp[2].value = 0.03

cals.export_values(cal_file)
# May need to get ro_freq again